# EDA, Datatvätt och transformering

In [1]:
import duckdb
conn = duckdb.connect()

In [2]:
%load_ext sql
%sql conn --alias duckdb

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

## Läs in rådata

In [3]:
%%sql
CREATE OR REPLACE TABLE raw_shows AS
SELECT *
FROM read_json(
    '../data/raw/show_details.jsonl',
    format = 'newline_delimited', # För JSONL som innehåller ett JSON-objekt per rad
    columns = {
        'id': 'INT',
        'name': 'VARCHAR',
        'original_name': 'VARCHAR',
        'first_air_date': 'VARCHAR', # VARCHAR eftersom DATE hade inte funkat om det finns tomma fält
        'last_air_date': 'VARCHAR',
        'status': 'VARCHAR',
        'type': 'VARCHAR',
        'in_production': 'BOOLEAN',
        'number_of_seasons': 'INT',
        'number_of_episodes': 'INT',
        'origin_country': 'VARCHAR[]',
        'original_language': 'VARCHAR',
        'popularity': 'DOUBLE',
        'vote_average': 'DOUBLE',
        'vote_count': 'INT',
        'fetched_at': 'VARCHAR',
        'genres': 'STRUCT(id INT, name VARCHAR)[]',
        'networks': 'STRUCT(id INT, name VARCHAR, origin_country VARCHAR)[]',
        'seasons': 'STRUCT(id INT, season_number INT, air_date VARCHAR, episode_count INT)[]'       
    }
);

Running query in 'duckdb'

1253 rows affected.

Count
1253


### Undersök råa tabellen och om det finns dubletter

In [4]:
%%sql
DESCRIBE raw_shows;

Running query in 'duckdb'

column_name,column_type,null,key,default,extra
id,INTEGER,YES,None,None,None
name,VARCHAR,YES,None,None,None
original_name,VARCHAR,YES,None,None,None
first_air_date,VARCHAR,YES,None,None,None
last_air_date,VARCHAR,YES,None,None,None
status,VARCHAR,YES,None,None,None
type,VARCHAR,YES,None,None,None
in_production,BOOLEAN,YES,None,None,None
number_of_seasons,INTEGER,YES,None,None,None
number_of_episodes,INTEGER,YES,None,None,None


Kollar om det finns dubletter:

In [5]:
%%sql
SELECT id, COUNT(*)
FROM raw_shows
GROUP BY id
HAVING COUNT(*) > 1

Running query in 'duckdb'

id,count_star()
